# J7Scope — zh/en J-lens walkthrough (M1)

End-to-end pass of the M1 correlational analysis on **Qwen2.5-7B-Instruct**:

1. fit one language-agnostic expected Jacobian `J_l`,
2. read out zh and en probe prompts at aligned positions,
3. compare the two direction sets (CKA / SVCCA / concept-level top-k overlap).

**Requirements:** GPU with ~20 GB VRAM (bf16). Run `pip install -e .` from the repo root first.

In [ ]:
import numpy as np
import torch

from j7scope.data import load_parallel_pairs
from j7scope.fitting import JLens, load_model
from j7scope.metrics import linear_cka, svcca
from j7scope.patching import concept_hit
from j7scope.viz import render_comparison_html

MODEL = "Qwen/Qwen2.5-7B-Instruct"
LAYER = 18  # of 28 decoder layers; sweep this later
DATA_DIR, RESULTS_DIR = "../data", "../results"

In [ ]:
model, tokenizer = load_model(MODEL)
pairs = load_parallel_pairs(DATA_DIR)
print(f"{len(pairs)} parallel pairs, {len(model.model.layers)} decoder layers")

## 1. Fit `J_l` once (language-agnostic)

`J_l` is a property of the model, estimated on a generic corpus and reused
unchanged for both languages. The tiny mixed corpus below is walkthrough-sized;
**TODO:** replace with a few hundred web-text sentences and crank `n_probes`
until readouts stabilize.

In [ ]:
fit_corpus = [
    "The committee postponed the decision until the next quarterly meeting.",
    "Rising sea levels threaten coastal cities across several continents.",
    "She spent the whole weekend repairing the old wooden boat.",
    "The experiment produced results that nobody on the team had predicted.",
    "委员会把这个决定推迟到了下一次季度会议。",
    "不断上升的海平面威胁着几大洲的沿海城市。",
    "她花了整个周末修理那艘旧木船。",
    "这次实验产生了团队里没有人预料到的结果。",
]

jlens = JLens(model, tokenizer, layer=LAYER)
jlens.estimate_jacobian(fit_corpus, n_probes=64)
print("J:", tuple(jlens.J.shape))

## 2. Read out one pair

Same concept (*deception*), same syntax, only the language differs. Do the
readouts agree at the concept level?

In [ ]:
pair = pairs["deception-01"]
for lang in ("zh", "en"):
    rec = pair[lang]
    top = jlens.readout(jlens.collect_residual(rec["text"]), k=10)
    print(f"[{lang}] {rec['text']}")
    print("     ", [t for t, _ in top], "\n")

## 3. M1 metrics over the whole corpus

Rows aligned by prompt id. We compare both raw residuals `H` and J-mapped
directions `Z = H Jᵀ`, against a shuffled-pairing null (same language
statistics, broken concept alignment).

In [ ]:
ids = sorted(pairs)
H = {lang: np.stack([jlens.collect_residual(pairs[i][lang]["text"]).float().cpu().numpy()
                     for i in ids])
     for lang in ("zh", "en")}
J = jlens.J.numpy()
Z = {lang: H[lang] @ J.T for lang in ("zh", "en")}

rng = np.random.default_rng(0)
perm = rng.permutation(len(ids))
for name, M in (("residuals H", H), ("J-mapped Z", Z)):
    print(f"{name}:  CKA={linear_cka(M['zh'], M['en']):.3f}"
          f"  (shuffled null={linear_cka(M['zh'], M['en'][perm]):.3f})"
          f"  SVCCA={svcca(M['zh'], M['en']):.3f}")

## 4. Concept-level readout table

Raw zh/en token strings never match literally; we score each side against its
own language's `expected` forms and export a side-by-side page.

In [ ]:
rows = []
for i in ids:
    zh, en = pairs[i]["zh"], pairs[i]["en"]
    top_zh = jlens.readout(jlens.collect_residual(zh["text"]), k=10)
    top_en = jlens.readout(jlens.collect_residual(en["text"]), k=10)
    hit_zh, hit_en = concept_hit(top_zh, zh["expected"]), concept_hit(top_en, en["expected"])
    rows.append(dict(id=i, concept=zh["concept"], zh_text=zh["text"], en_text=en["text"],
                     zh_topk=top_zh, en_topk=top_en, overlap=(hit_zh + hit_en) / 2))

both = sum(r["overlap"] == 1.0 for r in rows)
print(f"concept surfaced in BOTH languages: {both}/{len(rows)} pairs")
print("wrote", render_comparison_html(rows, f"{RESULTS_DIR}/m1_walkthrough.html"))

## Next: M2 (causal)

Once M1 shows signal, `j7scope.patching.patch_and_readout` transplants a
zh residual into the aligned en forward pass (and vice versa) and checks
whether the readout follows the **concept** or the **source language**:

```python
from j7scope.patching import patch_and_readout
res = patch_and_readout(jlens, src_prompt=pairs["manipulation-01"]["zh"]["text"],
                        tgt_prompt=pairs["manipulation-01"]["en"]["text"], patch_layer=12)
```